Settings

In [ ]:
DATASET = "all_cards"
NUM_EPOCHS = 10

Read data manifest

In [ ]:
import pandas as pd

df = pd.read_csv(f"data/{DATASET}_manifest.csv")
print(f"Total cards: {len(df)}")
print(df.head())
print(df["types"].value_counts().head(20))

Build dataset

In [ ]:
from PIL import Image
from torch.utils.data import Dataset
import torch


class CreatureDataset(Dataset):
    def __init__(self, df, all_types, transform=None):
        self.df = df.reset_index(drop=True)
        self.all_types = all_types
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        
        label = torch.zeros(len(self.all_types))
        types = row["types"].split("|") if isinstance(row["types"], str) else []
        for t in types:
            if t in self.all_types:
                label[self.all_types.index(t)] = 1.0
        
        return image, label

In [ ]:
all_types = sorted(set(
    t for types_str in df["types"] if isinstance(types_str, str) for t in types_str.strip().split("|")
))
print(all_types)
print(f"amount of creature types:", len(all_types))

In [ ]:
from torch.utils.data import random_split
from torch.utils.data import DataLoader
from torchvision import transforms
import random

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# create a fixed train/val split
indices = list(range(len(df)))
split = int(0.8 * len(df))
random.shuffle(indices)
train_indices = indices[:split]
val_indices = indices[split:]
train_dataset = CreatureDataset(df.iloc[train_indices], all_types, transform=train_transform)
val_dataset = CreatureDataset(df.iloc[val_indices], all_types, transform=val_transform)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

In [ ]:
from torch.utils.data import WeightedRandomSampler
from collections import Counter

# compute weight for each card
def compute_sample_weights(df, all_types, type_counts):
    weights = []
    for _, row in df.iterrows():
        if isinstance(row["types"], str):
            types = row["types"].split("|")
            # weight by inverse frequency of rarest type
            rarest_count = min(type_counts[t] for t in types)
            weights.append(1.0 / rarest_count)
        else:
            weights.append(1.0)
    return weights

type_counts = Counter(
    t for types_str in df["types"]
    if isinstance(types_str, str)
    for t in types_str.split("|")
)

weights = compute_sample_weights(df.iloc[train_indices], all_types, type_counts)
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=128, sampler=sampler, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=0)

In [ ]:
import matplotlib.pyplot as plt

types_sorted = sorted(type_counts.items(), key=lambda x: x[1], reverse=True)
labels, counts = zip(*types_sorted)

print(types_sorted)

plt.figure(figsize=(20, 6))
plt.bar(labels, counts)
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

Model

In [ ]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch.nn as nn

# Torch settings
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.enabled = False
torch.backends.cuda.matmul.allow_tf32 = True

# Model choice
model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, len(all_types))
print(model)

# Device choice
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model = model.to(device)
print(torch.__version__)
print(torch.version.cuda)

# we freeze all except the last layer
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

# Optimize
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam([
    {'params': model.features[6].parameters(), 'lr': 1e-5},
    {'params': model.features[7].parameters(), 'lr': 2e-5},
    {'params': model.features[8].parameters(), 'lr': 5e-5},
    {'params': model.classifier.parameters(), 'lr': 1e-4}
])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

Training

In [ ]:
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler
from pathlib import Path

for epoch in range(1, NUM_EPOCHS + 1):
    # train
    model.train()
    train_loss = 0.0
    scaler = torch.amp.GradScaler('cuda')
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} train"):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    # validate
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} val"):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        print(f"Epoch {epoch}/{NUM_EPOCHS} — train loss: {avg_train_loss:.4f} — val loss: {avg_val_loss:.4f}")
    
    # unfreeze layers at threshold
    if epoch == 1 * NUM_EPOCHS // 5:
        print("unfreezing layer features[8]")
        for param in model.features[8].parameters():
            param.requires_grad = True
    if epoch == 2 * NUM_EPOCHS // 5:
        print("unfreezing layer features[7]")
        for param in model.features[7].parameters():
            param.requires_grad = True
    if epoch == 3 * NUM_EPOCHS // 5:
        print("unfreezing layer features[6]")
        for param in model.features[6].parameters():
            param.requires_grad = True

    # save model after epoch
    path = Path(f"models/checkpoint_epoch_{epoch}.pth")
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        'model_state_dict': model.state_dict(),
        'all_types': all_types,
        'epoch': epoch,
        'val_loss': avg_val_loss,
    }, data)